# Data Science for Business - Multilayer Perceptron (MLP) on Ames Housing with PyTorch

*Session 8 · ISLP 10.1–10.2 · 10.7.1–10.7.3 (backpropagation, SGD, dropout)*

Many PyTorch examples use the add-on `pytorch_lightning`, which hides the training loop. Here, we write the training loop by hand to see every step; only the syntax differs, the model is the same.

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
# Install packages that are not pre-installed on Colab
# !pip install torchinfo

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchinfo import summary

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
plt.style.use('fivethirtyeight')

Use a GPU if one is available: an NVIDIA GPU (`cuda`, e.g., on Colab), Apple silicon (`mps`), or else the CPU.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('Using device:', device)

## Problem description

Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence. With 76 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, this dataset challenges you to predict the final price of each home. More: <https://www.kaggle.com/c/house-prices-advanced-regression-techniques>

## Load data

Load training data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_08/ameshousing.csv')

In [ ]:
data.head()

## Prepare data

First, we will remove some columns that are not useful for our task.

In [ ]:
data = data.drop(['YrSold', 'MoSold', 'SaleCondition', 'SaleType'], axis=1)

`MSSubClass` (type of dwelling) is stored as a number, but its values are codes for categories (e.g., 20 = 1-story house built 1946 or later, 60 = 2-story house built 1946 or later; see the data description). We therefore add the prefix `MSSC` to all values, which turns the variable into a string, so that it is treated as categorical.

In [ ]:
data["MSSubClass"] = "MSSC" + data["MSSubClass"].astype(str)

Next, we will split the data into predictors (*X*) and response (*y*) and into training (*X_train, y_train*) and test (*X_test, y_test*) sets.

In [ ]:
X = data.drop("SalePrice", axis=1)
y = data["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

To monitor the training of the network, we set aside 20% of the training data as a validation set (ISLP 5.1.1). The network never trains on it; we only compute the loss on it after each epoch.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
print(X_train.shape, X_val.shape)

Finally, we will prepare the predictors for the network. It is important to use only information from the training set for this (e.g., the categories and the means and standard deviations), and then mechanistically repeat these steps on the validation and test sets.

Typically, feature engineering depends strongly on the datatype of the variables. Hence, we will first determine which variables are categorical and which are numerical. Subsequently, we will transform these variables separately.

In [ ]:
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns
numerical_features = X_train.select_dtypes(exclude=['object', 'string']).columns

The categorical variables must be transformed into numerical representations, e.g., by one-hot encoding them. `handle_unknown='ignore'` encodes categories that do not appear in the training set as all zeros instead of raising an error.

In [ ]:
onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

The numerical variables will be standardized, that is, we will subtract the mean and divide by the standard deviation. Neural networks are trained by gradient descent, which works much better when all inputs are on a similar scale.

In [ ]:
scaler = StandardScaler()

A `ColumnTransformer` applies each transformer to its own columns and puts the results side by side. `.set_output(transform='pandas')` returns a DataFrame, so the feature names stay readable. This is the same preprocessor as in the scikit-learn notebook (`ames_mlp_scikit.ipynb`).

In [ ]:
preprocessor = ColumnTransformer(
                transformers=[
                        ('num', scaler, numerical_features),
                        ('cat', onehot, categorical_features)
                    ], verbose_feature_names_out=False
                ).set_output(transform='pandas')

In the scikit-learn notebook, the preprocessor and the model sit together in a `Pipeline`. A PyTorch model cannot be a step of a scikit-learn pipeline, so here we apply the preprocessor ourselves: `fit_transform` learns the categories, means and standard deviations from the training data and transforms it; `transform` applies the same, already fitted, steps to the validation and test data.

In [ ]:
X_train = preprocessor.fit_transform(X_train)
X_val = preprocessor.transform(X_val)
X_test = preprocessor.transform(X_test)

In [ ]:
X_train.shape

The response needs scaling, too: sale prices are around 180,000, so the squared-error loss on the raw prices is in the order of $10^{10}$ and the gradients are poorly scaled. We therefore standardize the response with the mean and standard deviation of the training set, and transform the predictions back to dollars at the end.

In [ ]:
y_mean, y_std = y_train.mean(), y_train.std()

y_train_scaled = (y_train - y_mean) / y_std
y_val_scaled = (y_val - y_mean) / y_std

## Neural network

### Data loaders

The first thing we have to do is to transform the data into PyTorch tensors. This is done by the `torch.tensor` function. The response gets an extra dimension (`unsqueeze(1)`), because the network outputs one column.

In [ ]:
X_train_tensor = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
X_val_tensor = torch.tensor(X_val.to_numpy(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.to_numpy(), dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled.to_numpy(), dtype=torch.float32).unsqueeze(1)
y_val_tensor = torch.tensor(y_val_scaled.to_numpy(), dtype=torch.float32).unsqueeze(1)

Next, we create a data loader for the training data, using the `torch.utils.data.DataLoader` class. The data loader allows to read the data in a streaming fashion during training. This is actually not needed for our tiny dataset, but normally one uses PyTorch with much larger datasets that do not fit into main memory.

The data loader shuffles the training data and serves it in minibatches of 32 houses. Each minibatch is used for one gradient step, so one epoch (one pass over the training data) consists of about 1,875/32 ≈ 59 steps (ISLP 10.7.2). The seeded `generator` makes the shuffling reproducible.

In [ ]:
batch_size = 32

train_loader = DataLoader(
    dataset=TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=batch_size,
    shuffle=True,
    generator=torch.Generator().manual_seed(42)
)
print('Minibatches per epoch:', len(train_loader))

### Network architecture

Now we specify the architecture of the neural network. We will use a feedforward neural network with three hidden layers (ISLP 10.2). The input layer has as many units as we have predictors, the hidden layers have 512, 256 and 128 hidden units with ReLU activation, and the output layer has one unit, as we want to do regression.

After each hidden layer, we apply dropout: during training, 20% of the hidden units are randomly set to zero in each step, a form of regularisation (ISLP 10.7.3). `model.eval()` switches dropout off for predictions.

To specify the architecture, we create a class that inherits from `torch.nn.Module` and has two standard methods. In the `__init__` method, we specify the types of layers we want to use as building blocks. The `forward` method specifies how these building blocks are connected.

In [ ]:
class MLPModel(nn.Module):
    def __init__(self, input_dim):
        super(MLPModel, self).__init__()
        self.hidden1 = nn.Linear(input_dim, 512)
        self.hidden2 = nn.Linear(512, 256)
        self.hidden3 = nn.Linear(256, 128)
        self.output = nn.Linear(128, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.relu(self.hidden1(x))
        x = self.dropout(x)
        x = self.relu(self.hidden2(x))
        x = self.dropout(x)
        x = self.relu(self.hidden3(x))
        x = self.dropout(x)
        x = self.output(x)
        return x

Let's create an instance of the neural network, move it to the device, and look at its architecture with `torchinfo`.

In [ ]:
input_dim = X_train.shape[1]
model = MLPModel(input_dim).to(device)

In [ ]:
summary(model, input_size=(batch_size, input_dim), device=device)

### Training loop

We are now ready to train the neural network. We first specify the loss function, which is the mean squared error loss, and the optimizer, Adam, a popular variant of stochastic gradient descent with learning rate ρ = 0.001 (`lr`).

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

We then train the network. The training loop consists of two nested loops:

- The **outer loop** runs over **epochs**. One epoch is one full pass over all training houses. We train for 100 epochs.
- The **inner loop** runs over the **minibatches** that the data loader serves (32 houses each, about 59 per epoch). For every minibatch, the weights are updated once.

In each epoch, the following happens:

1. `model.train()` puts the model in *training mode*. This switches dropout on.
2. For each minibatch from `train_loader` (inner loop):
   1. **Move the data to the device** (`.to(device)`), i.e., to the GPU if we use one. Model and data must be on the same device.
   2. **Reset the gradients** with `optimizer.zero_grad()`. PyTorch adds up gradients from previous steps by default, so we have to clear them before computing new ones.
   3. **Forward pass:** `model(batch_X)` feeds the minibatch through the network and returns the predicted (standardized) prices.
   4. **Compute the loss:** `criterion(predictions, batch_y)` computes the mean squared error between predictions and true prices for this minibatch.
   5. **Backward pass:** `loss.backward()` computes the gradient of the loss with respect to every weight in the network (backpropagation, ISLP 10.7.1). The gradient tells us in which direction each weight would have to change to increase the loss.
   6. **Update the weights:** `optimizer.step()` moves every weight a small step in the opposite direction of its gradient, i.e., in the direction that decreases the loss. The step size is controlled by the learning rate ρ (ISLP 10.7.1–10.7.2).
   7. **Keep track of the loss:** `loss.item()` turns the loss into a plain Python number, which we add up over the minibatches.
3. After all minibatches, we store the **average training loss** of the epoch.
4. **Validation:** `model.eval()` puts the model in *evaluation mode* (dropout off), and `torch.no_grad()` tells PyTorch that we don't need gradients here (we only want to measure, not learn). We predict the validation houses and store the validation loss. The validation set is never used to update the weights.
5. Every 10 epochs, we print both losses, so we can follow the progress.

In [ ]:
epochs = 100
train_losses = []   # average training loss per epoch
val_losses = []     # validation loss per epoch

for epoch in range(epochs):                                   # outer loop: one pass over the training data per epoch
    # Training
    model.train()                                             # training mode: dropout on
    epoch_loss = 0
    for batch_X, batch_y in train_loader:                     # inner loop: one minibatch of 32 houses per step
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)   # move data to the same device as the model
        optimizer.zero_grad()                                 # reset gradients from the previous step
        predictions = model(batch_X)                          # forward pass: predict the minibatch
        loss = criterion(predictions, batch_y)                # mean squared error on this minibatch
        loss.backward()                                       # backward pass: compute gradients (backpropagation)
        optimizer.step()                                      # update weights: one step against the gradient
        epoch_loss += loss.item()                             # add up the loss of this minibatch
    train_losses.append(epoch_loss / len(train_loader))       # average training loss of this epoch

    # Validation
    model.eval()                                              # evaluation mode: dropout off
    with torch.no_grad():                                     # no gradients needed, we only measure
        val_loss = criterion(model(X_val_tensor.to(device)), y_val_tensor.to(device))
    val_losses.append(val_loss.item())

    if (epoch + 1) % 10 == 0:                                 # print progress every 10 epochs
        print(f'Epoch {epoch + 1}/{epochs}, Training loss: {train_losses[-1]:.4f}, Validation loss: {val_losses[-1]:.4f}')

Let's plot the training and validation loss per epoch (compare ISLP Figure 10.18). Note that the training loss is computed with dropout switched on, so it is not directly comparable to the validation loss.

In [ ]:
plt.plot(range(1, epochs + 1), train_losses, label='Training')
plt.plot(range(1, epochs + 1), val_losses, label='Validation')
plt.xlabel('Epoch')
plt.ylabel('MSE (standardized response)')
plt.legend()
plt.show()

Here, the validation loss levels off after roughly 10 to 20 epochs, while the training loss keeps decreasing: the additional epochs mainly fit the training data more closely. If the validation loss starts to increase, the network overfits; we could then stop training earlier (early stopping, ISLP 10.7.2).

### Predictions and evaluation

Finally, we evaluate the neural network on the test set. Therefore, we set the model to evaluation mode, so that the dropout layers are not active. The network predicts standardized prices, so we transform the predictions back to dollars: multiply by the standard deviation and add the mean.

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_tensor.to(device)).cpu().numpy().ravel()

y_pred = y_pred_scaled * y_std + y_mean

Let's compute R² and RMSE in dollars.

In [ ]:
r2_test = r2_score(y_test, y_pred)
rmse_test = root_mean_squared_error(y_test, y_pred)
print('R2 on test set:', round(r2_test, 2))
print('RMSE on test set:', round(rmse_test, 2))

Compare the result with the scikit-learn MLP and the lasso baseline from the other Session 8 notebook (same train/test split). Note that this network was trained on 80% of the training data only, because we held out a validation set.

## Your turn!

**Your task:** Try to improve the test RMSE. You can try the following:

- Change the number of epochs (look at the validation loss curve first)
- Change the number of hidden units or hidden layers
- Vary the dropout rate
- Train on the log of the price instead of the standardized price
- ...

In [ ]:
# YOUR CODE HERE